In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score

for d in [Path("."), Path("data"), Path("results/pre_v5_backup")]:
    f = d / "probs_test.npy"
    if not f.exists():
        print(f"{str(d):28s} 없음")
        continue
    pt = np.load(f)
    rt = pd.read_csv(d / "preds_test.csv") if (d / "preds_test.csv").exists() \
         else pd.read_csv("preds_test.csv")
    acc = accuracy_score(rt.true_idx.values, pt.argmax(1))
    tag = "v5 ✔" if abs(acc - 0.9859) < 1e-4 else "v3"
    print(f"{str(d):28s} accuracy {acc:.4f}  {tag}")

.                            없음
data                         accuracy 0.9859  v5 ✔
results\pre_v5_backup        없음


In [3]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DDIR = Path("data")                 # probs/preds/split 위치
OUT  = Path("outputs/presentation")
TAU  = 0.53

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
OUT.mkdir(parents=True, exist_ok=True)

sp = pd.read_csv(DDIR / "split_hj.csv")
rt = pd.read_csv(DDIR / "preds_test.csv")
pt = np.load(DDIR / "probs_test.npy")
classes = sorted(sp["class"].unique())

te = sp[sp.split == "test"].reset_index(drop=True)
assert (te.label_idx.values == rt.true_idx.values).all(), "순서 불일치"

# ── 데이터셋 루트 자동 탐색 ──────────────────────────────────
CANDIDATES = [Path("garbage_classification"),
              Path("data/garbage_classification"),
              Path("../garbage_classification"),
              Path("../data/garbage_classification"),
              Path("../archive/garbage_classification")]
probe = te.iloc[0]
DATA = None
for c in CANDIDATES:
    if (c / probe["class"] / probe.fname).exists():
        DATA = c
        break
if DATA is None:
    print("데이터셋 폴더를 못 찾았습니다. 아래 중 실제 경로를 DATA 에 직접 넣으세요:")
    for c in CANDIDATES:
        print("   ", c.resolve(), "->", c.exists())
    raise SystemExit
print("데이터셋 경로:", DATA.resolve())
#

데이터셋 경로: C:\Users\KDT-17\Documents\17-DeepLearning\data\garbage_classification


In [4]:
print("df 존재:", 'df' in dir())
print("grid 존재:", 'grid' in dir())
print("DATA:", DATA if 'DATA' in dir() else "없음")

df 존재: True
grid 존재: True
DATA: ..\data\garbage_classification


In [5]:
wp = df[((df.true == "white-glass") & (df.pred == "plastic")) |
        ((df.true == "plastic") & (df.pred == "white-glass"))]
wp = wp.sort_values("conf", ascending=False).head(6)
grid(list(wp.itertuples()),
     "가장 많이 헷갈린 조합 — white-glass 와 plastic",
     "test 오분류 33건 중 9건이 이 조합 · 투명 유리병과 투명 페트병",
     "img_confusion_pairs.png")

hold = df[df.conf < TAU].sort_values("conf")
grid(list(hold.itertuples()),
     f"사람에게 넘어가는 {len(hold)}장",
     f"확신도 {TAU} 미만 · 전체의 {len(hold)/len(df)*100:.1f}% · 나머지는 98.8% 정확도로 자동 분류",
     "img_low_confidence.png", ncol=4)

print("완료 —", OUT.resolve())
for f in sorted(OUT.glob("*.png")):
    print(" ", f.name, f"{f.stat().st_size/1024:.0f} KB")

[저장] img_confusion_pairs.png (6장)
[저장] img_low_confidence.png (8장)
완료 — C:\Users\KDT-17\Documents\17-DeepLearning\Garbage_Classification\outputs\presentation
  img_confusion_pairs.png 566 KB
  img_low_confidence.png 946 KB


In [8]:
def grid2(rows, filename, ncol=3):
    nrow = -(-len(rows) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.5 * nrow))
    fig.patch.set_facecolor(PAPER)
    axes = np.atleast_1d(axes).ravel()

    for ax, r in zip(axes, rows):
        ax.imshow(Image.open(DATA / r.cls / r.fname).convert("RGB"))
        ax.set_xticks([])
        ax.set_yticks([])
        ok = r.true == r.pred
        c = GREEN if ok else ORANGE
        cap = r.true + "  (정답)" if ok else r.true + "  →  " + r.pred
        cap = cap + "   ·   확신도 " + format(r.conf, ".2f")
        ax.set_title(cap, fontsize=11, color=c, fontweight="bold", pad=6)
        for sp_ in ax.spines.values():
            sp_.set_color(c)
            sp_.set_linewidth(2.5)

    for ax in axes[len(rows):]:
        ax.axis("off")

    fig.subplots_adjust(top=0.94, left=0.02, right=0.98, bottom=0.02, wspace=0.10, hspace=0.22)
    fig.savefig(OUT / filename, dpi=180, facecolor=PAPER)
    plt.close(fig)
    print("[저장]", filename, len(rows), "장")


grid2(list(wp.itertuples()), "img_confusion_pairs.png", ncol=3)
grid2(list(hold.itertuples()), "img_low_confidence.png", ncol=4)

[저장] img_confusion_pairs.png 6 장
[저장] img_low_confidence.png 8 장
